In [ ]:
# In[1]:

import pandas as pd
import pytz
from datetime import datetime

tz = pytz.timezone('Asia/Shanghai')

def epoch_to_shanghai(ts):
    # handle if ts is NaN or None
    try:
        return datetime.fromtimestamp(int(ts), tz).isoformat()
    except Exception:
        return None

# Load files
df_metric = pd.read_csv('metric.csv')
df_trace = pd.read_csv('trace.csv')
df_log = pd.read_csv('log.csv')
df_err = pd.read_csv('error_logs.csv')

# Helper to create summary for metric/trace/log
def summarize_df(df, name_col):
    total_rows = int(df.shape[0])
    unique_cmdb = df['cmdb_id'].dropna().unique()
    unique_cmdb_list = list(map(str, unique_cmdb[:50]))
    unique_cmdb_count = int(len(unique_cmdb))
    unique_names = df[name_col].dropna().unique() if name_col in df.columns else []
    unique_name_list = list(map(str, unique_names[:50]))
    unique_name_count = int(len(unique_names))
    if total_rows > 0:
        min_ts = int(df['timestamp'].min())
        max_ts = int(df['timestamp'].max())
    else:
        min_ts = None
        max_ts = None
    min_dt = epoch_to_shanghai(min_ts) if min_ts is not None else None
    max_dt = epoch_to_shanghai(max_ts) if max_ts is not None else None
    top5 = df.sort_values('timestamp', ascending=True).head(5)
    # select only requested columns
    top5_sel = top5[['timestamp', 'cmdb_id', name_col, 'value']].copy()
    # convert timestamp to epoch and human readable in a compact column
    top5_sel['ts_shanghai'] = top5_sel['timestamp'].apply(epoch_to_shanghai)
    top5_sel = top5_sel.reset_index(drop=True)
    return {
        'total_rows': total_rows,
        'unique_cmdb_count': unique_cmdb_count,
        'unique_cmdb_list': unique_cmdb_list,
        'unique_name_count': unique_name_count,
        'unique_name_list': unique_name_list,
        'min_timestamp': min_ts,
        'max_timestamp': max_ts,
        'min_timestamp_shanghai': min_dt,
        'max_timestamp_shanghai': max_dt,
        'top5_rows': top5_sel
    }

# Summaries
metric_summary = summarize_df(df_metric, 'kpi_name')
trace_summary = summarize_df(df_trace, 'trace_name')
log_summary = summarize_df(df_log, 'log_name')

# error_logs has no value column; use message
def summarize_error_logs(df):
    total_rows = int(df.shape[0])
    unique_cmdb = df['cmdb_id'].dropna().unique()
    unique_cmdb_list = list(map(str, unique_cmdb[:50]))
    unique_cmdb_count = int(len(unique_cmdb))
    # For error_logs, there is no KPI name; treat 'message' as the name field
    unique_messages = df['message'].dropna().unique()
    unique_message_list = list(map(str, unique_messages[:50]))
    unique_message_count = int(len(unique_messages))
    if total_rows > 0:
        min_ts = int(df['timestamp'].min())
        max_ts = int(df['timestamp'].max())
    else:
        min_ts = None
        max_ts = None
    min_dt = epoch_to_shanghai(min_ts) if min_ts is not None else None
    max_dt = epoch_to_shanghai(max_ts) if max_ts is not None else None
    top5 = df.sort_values('timestamp', ascending=True).head(5)
    top5_sel = top5[['timestamp', 'cmdb_id', 'message']].copy()
    top5_sel['ts_shanghai'] = top5_sel['timestamp'].apply(epoch_to_shanghai)
    top5_sel = top5_sel.reset_index(drop=True)
    return {
        'total_rows': total_rows,
        'unique_cmdb_count': unique_cmdb_count,
        'unique_cmdb_list': unique_cmdb_list,
        'unique_message_count': unique_message_count,
        'unique_message_list': unique_message_list,
        'min_timestamp': min_ts,
        'max_timestamp': max_ts,
        'min_timestamp_shanghai': min_dt,
        'max_timestamp_shanghai': max_dt,
        'top5_rows': top5_sel
    }

error_logs_summary = summarize_error_logs(df_err)

summary = {
    'metric': metric_summary,
    'trace': trace_summary,
    'log': log_summary,
    'error_logs': error_logs_summary
}

summary

```
Out[1]:
```
Summary of the loaded telemetry files (compact):

Metric:
- Total rows: 144,252
- Unique cmdb_id: 59 (examples, first values): adservice, adservice-0, adservice-1, adservice-2, adservice2, adservice2-0, cartservice, cartservice-0, cartservice-1, cartservice-2, ...
- Unique KPI names: 1,152 (examples): app.grpc.count, app.grpc.mrt, app.grpc.rr, app.grpc.sr, app.http.count, runtime.java_lang_ClassLoading_LoadedClassCount, runtime.java_lang_GarbageCollector_LastGcInfo_duration, ...
- Timestamp range (Asia/Shanghai / UTC+8): 2022-03-20T12:30:00 to 2022-03-20T12:59:00
- Top 5 earliest rows (timestamp, cmdb_id, kpi_name, value):
  1. 2022-03-20T12:30:00, adservice, runtime.java_lang_GarbageCollector_LastGcInfo_... , 327680.0
  2. 2022-03-20T12:30:00, adservice, runtime.java_lang_GarbageCollector_LastGcInfo_... , 327680.0
  3. 2022-03-20T12:30:00, adservice, runtime.java_lang_GarbageCollector_LastGcInfo_... , 7012352.0
  4. 2022-03-20T12:30:00, adservice, runtime.java_lang_GarbageCollector_LastGcInfo_... , 7012352.0
  5. 2022-03-20T12:30:00, adservice, runtime.java_lang_GarbageCollector_LastGcInfo_... , 251658240.0

Trace:
- Total rows: 36,656
- Unique cmdb_id: 40 (examples): adservice-0, adservice-1, adservice-2, adservice2-0, cartservice-0, cartservice-1, ...
- Unique trace names: 200 (examples): trace.from_frontend-0.duration_mean, trace.from_frontend-0.duration_p95, trace.from_frontend-0.error_rate, trace.self.duration_mean, trace.self.error_rate, trace.to_cartservice-0.duration_mean, ...
- Timestamp range (Asia/Shanghai / UTC+8): 2022-03-20T12:30:00 to 2022-03-20T12:59:00
- Top 5 earliest rows (timestamp, cmdb_id, trace_name, value):
  1. 2022-03-20T12:30:00, adservice-1, trace.from_frontend-1.duration_mean, 0.000013
  2. 2022-03-20T12:30:00, adservice-1, trace.from_frontend-1.duration_p95, 0.000014
  3. 2022-03-20T12:30:00, adservice-1, trace.from_frontend-1.error_rate, 0.0
  4. 2022-03-20T12:30:00, adservice-1, trace.from_frontend-1.row_count, 21.0
  5. 2022-03-20T12:30:00, adservice-1, trace.from_frontend-2.duration_mean, 0.000013

Log:
- Total rows: 1,814
- Unique cmdb_id: 31 (examples): adservice-0, adservice-1, adservice-2, cartservice-0, cartservice-1, checkoutservice-0, ...
- Unique log names: 2 — log.error_count, log.row_count
- Timestamp range (Asia/Shanghai / UTC+8): 2022-03-20T12:30:00 to 2022-03-20T12:59:00
- Top 5 earliest rows (timestamp, cmdb_id, log_name, value):
  1. 2022-03-20T12:30:00, adservice-0, log.row_count, 1528.0
  2. 2022-03-20T12:30:00, adservice-0, log.error_count, 0.0
  3. 2022-03-20T12:30:00, checkoutservice-1, log.row_count, 85.0
  4. 2022-03-20T12:30:00, checkoutservice-1, log.error_count, 0.0
  5. 2022-03-20T12:30:00, checkoutservice-0, log.row_count, 79.0

Error logs:
- Total rows: 1,994
- Unique cmdb_id: 3 — frontend-1, frontend-2, frontend-0
- Unique messages: 1 — "severity: warning, message: failed to retrieve ads"
- Timestamp range (Asia/Shanghai / UTC+8): 2022-03-20T12:30:00 to 2022-03-20T12:59:59
- Top 5 earliest rows (timestamp, cmdb_id, message):
  1. 2022-03-20T12:30:00, frontend-1, severity: warning, message: failed to retrieve ads
  2. 2022-03-20T12:30:00, frontend-2, severity: warning, message: failed to retrieve ads
  3. 2022-03-20T12:30:03, frontend-0, severity: warning, message: failed to retrieve ads
  4. 2022-03-20T12:30:04, frontend-1, severity: warning, message: failed to retrieve ads
  5. 2022-03-20T12:30:05, frontend-0, severity: warning, message: failed to retrieve ads

If you want, I can expand any of these lists (up to the requested limits) or provide additional filtered views.

The original code execution output of IPython Kernel is also provided below for reference:

{'metric': {'total_rows': 144252, 'unique_cmdb_count': 59, 'unique_cmdb_list': ['adservice', 'adservice-0', 'adservice-1', 'adservice-2', 'adservice2', 'adservice2-0', 'cartservice', 'cartservice-0', 'cartservice-1', 'cartservice-2', 'cartservice2-0', 'checkoutservice', 'checkoutservice-0', 'checkoutservice-1', 'checkoutservice-2', 'checkoutservice2-0', 'currencyservice', 'currencyservice-0', 'currencyservice-1', 'currencyservice-2', 'currencyservice2-0', 'emailservice', 'emailservice-0', 'emailservice-1', 'emailservice-2', 'emailservice2-0', 'frontend', 'frontend-0', 'frontend-1', 'frontend-2', 'frontend2-0', 'node-1', 'node-2', 'node-3', 'node-4', 'node-5', 'node-6', 'paymentservice', 'paymentservice-0', 'paymentservice-1', 'paymentservice-2', 'paymentservice2-0', 'productcatalogservice', 'productcatalogservice-0', 'productcatalogservice-1', 'productcatalogservice-2', 'productcatalogservice2-0', 'recommendationservice', 'recommendationservice-0', 'recommendationservice-1'], 'unique_name_count': 1152, 'unique_name_list': ['app.grpc.count', 'app.grpc.mrt', 'app.grpc.rr', 'app.grpc.sr', 'app.http.count', 'app.http.mrt', 'app.http.rr', 'app.http.sr', 'runtime.java_lang_ClassLoading_LoadedClassCount', 'runtime.java_lang_ClassLoading_TotalLoadedClassCount', 'runtime.java_lang_ClassLoading_UnloadedClassCount', 'runtime.java_lang_ClassLoading_Verbose', 'runtime.java_lang_Compilation_CompilationTimeMonitoringSupported', 'runtime.java_lang_Compilation_TotalCompilationTime', 'runtime.java_lang_GarbageCollector_CollectionCount.Copy', 'runtime.java_lang_GarbageCollector_CollectionCount.MarkSweepCompact', 'runtime.java_lang_GarbageCollector_CollectionTime.Copy', 'runtime.java_lang_GarbageCollector_CollectionTime.MarkSweepCompact', 'runtime.java_lang_GarbageCollector_LastGcInfo_GcThreadCount.Copy', 'runtime.java_lang_GarbageCollector_LastGcInfo_GcThreadCount.MarkSweepCompact', 'runtime.java_lang_GarbageCollector_LastGcInfo_duration.Copy', 'runtime.java_lang_GarbageCollector_LastGcInfo_duration.MarkSweepCompact', 'runtime.java_lang_GarbageCollector_LastGcInfo_endTime.Copy', 'runtime.java_lang_GarbageCollector_LastGcInfo_endTime.MarkSweepCompact', 'runtime.java_lang_GarbageCollector_LastGcInfo_id.Copy', 'runtime.java_lang_GarbageCollector_LastGcInfo_id.MarkSweepCompact', 'runtime.java_lang_GarbageCollector_LastGcInfo_memoryUsageAfterGc_committed.Code_Cache.Copy', 'runtime.java_lang_GarbageCollector_LastGcInfo_memoryUsageAfterGc_committed.Code_Cache.MarkSweepCompact', 'runtime.java_lang_GarbageCollector_LastGcInfo_memoryUsageAfterGc_committed.Compressed_Class_Space.Copy', 'runtime.java_lang_GarbageCollector_LastGcInfo_memoryUsageAfterGc_committed.Compressed_Class_Space.MarkSweepCompact', 'runtime.java_lang_GarbageCollector_LastGcInfo_memoryUsageAfterGc_committed.Eden_Space.Copy', 'runtime.java_lang_GarbageCollector_LastGcInfo_memoryUsageAfterGc_committed.Eden_Space.MarkSweepCompact', 'runtime.java_lang_GarbageCollector_LastGcInfo_memoryUsageAfterGc_committed.Metaspace.Copy', 'runtime.java_lang_GarbageCollector_LastGcInfo_memoryUsageAfterGc_committed.Metaspace.MarkSweepCompact', 'runtime.java_lang_GarbageCollector_LastGcInfo_memoryUsageAfterGc_committed.Survivor_Space.Copy', 'runtime.java_lang_GarbageCollector_LastGcInfo_memoryUsageAfterGc_committed.Survivor_Space.MarkSweepCompact', 'runtime.java_lang_GarbageCollector_LastGcInfo_memoryUsageAfterGc_committed.Tenured_Gen.Copy', 'runtime.java_lang_GarbageCollector_LastGcInfo_memoryUsageAfterGc_committed.Tenured_Gen.MarkSweepCompact', 'runtime.java_lang_GarbageCollector_LastGcInfo_memoryUsageAfterGc_init.Code_Cache.Copy', 'runtime.java_lang_GarbageCollector_LastGcInfo_memoryUsageAfterGc_init.Code_Cache.MarkSweepCompact', 'runtime.java_lang_GarbageCollector_LastGcInfo_memoryUsageAfterGc_init.Compressed_Class_Space.Copy', 'runtime.java_lang_GarbageCollector_LastGcInfo_memoryUsageAfterGc_init.Compressed_Class_Space.MarkSweepCompact', 'runtime.java_lang_GarbageCollector_LastGcInfo_memoryUsageAfterGc_init.Eden_Space.Copy', 'runtime.java_lang_GarbageCollector_LastGcInfo_memoryUsageAfterGc_init.Eden_Space.MarkSweepCompact', 'runtime.java_lang_GarbageCollector_LastGcInfo_memoryUsageAfterGc_init.Metaspace.Copy', 'runtime.java_lang_GarbageCollector_LastGcInfo_memoryUsageAfterGc_init.Metaspace.MarkSweepCompact', 'runtime.java_lang_GarbageCollector_LastGcInfo_memoryUsageAfterGc_init.Survivor_Space.Copy', 'runtime.java_lang_GarbageCollector_LastGcInfo_memoryUsageAfterGc_init.Survivor_Space.MarkSweepCompact', 'runtime.java_lang_GarbageCollector_LastGcInfo_memoryUsageAfterGc_init.Tenured_Gen.Copy', 'runtime.java_lang_GarbageCollector_LastGcInfo_memoryUsageAfterGc_init.Tenured_Gen.MarkSweepCompact'], 'min_timestamp': 1647750600, 'max_timestamp': 1647752340, 'min_timestamp_shanghai': '2022-03-20T12:30:00+08:00', 'max_timestamp_shanghai': '2022-03-20T12:59:00+08:00', 'top5_rows':     timestamp    cmdb_id                                           kpi_name        value                ts_shanghai
0  1647750600  adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...     327680.0  2022-03-20T12:30:00+08:00
1  1647750600  adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...     327680.0  2022-03-20T12:30:00+08:00
2  1647750600  adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...    7012352.0  2022-03-20T12:30:00+08:00
3  1647750600  adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...    7012352.0  2022-03-20T12:30:00+08:00
4  1647750600  adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...  251658240.0  2022-03-20T12:30:00+08:00}, 'trace': {'total_rows': 36656, 'unique_cmdb_count': 40, 'unique_cmdb_list': ['adservice-0', 'adservice-1', 'adservice-2', 'adservice2-0', 'cartservice-0', 'cartservice-1', 'cartservice-2', 'cartservice2-0', 'checkoutservice-0', 'checkoutservice-1', 'checkoutservice-2', 'checkoutservice2-0', 'currencyservice-0', 'currencyservice-1', 'currencyservice-2', 'currencyservice2-0', 'emailservice-0', 'emailservice-1', 'emailservice-2', 'emailservice2-0', 'frontend-0', 'frontend-1', 'frontend-2', 'frontend2-0', 'paymentservice-0', 'paymentservice-1', 'paymentservice-2', 'paymentservice2-0', 'productcatalogservice-0', 'productcatalogservice-1', 'productcatalogservice-2', 'productcatalogservice2-0', 'recommendationservice-0', 'recommendationservice-1', 'recommendationservice-2', 'recommendationservice2-0', 'shippingservice-0', 'shippingservice-1', 'shippingservice-2', 'shippingservice2-0'], 'unique_name_count': 200, 'unique_name_list': ['trace.from_frontend-0.duration_mean', 'trace.from_frontend-0.duration_p95', 'trace.from_frontend-0.error_rate', 'trace.from_frontend-0.row_count', 'trace.from_frontend-1.duration_mean', 'trace.from_frontend-1.duration_p95', 'trace.from_frontend-1.error_rate', 'trace.from_frontend-1.row_count', 'trace.from_frontend-2.duration_mean', 'trace.from_frontend-2.duration_p95', 'trace.from_frontend-2.error_rate', 'trace.from_frontend-2.row_count', 'trace.from_frontend2-0.duration_mean', 'trace.from_frontend2-0.duration_p95', 'trace.from_frontend2-0.error_rate', 'trace.from_frontend2-0.row_count', 'trace.from_checkoutservice-0.duration_mean', 'trace.from_checkoutservice-0.duration_p95', 'trace.from_checkoutservice-0.error_rate', 'trace.from_checkoutservice-0.row_count', 'trace.from_checkoutservice-1.duration_mean', 'trace.from_checkoutservice-1.duration_p95', 'trace.from_checkoutservice-1.error_rate', 'trace.from_checkoutservice-1.row_count', 'trace.from_checkoutservice-2.duration_mean', 'trace.from_checkoutservice-2.duration_p95', 'trace.from_checkoutservice-2.error_rate', 'trace.from_checkoutservice-2.row_count', 'trace.self.duration_mean', 'trace.self.duration_p95', 'trace.self.error_rate', 'trace.self.row_count', 'trace.from_checkoutservice2-0.duration_mean', 'trace.from_checkoutservice2-0.duration_p95', 'trace.from_checkoutservice2-0.error_rate', 'trace.from_checkoutservice2-0.row_count', 'trace.from_root.duration_mean', 'trace.from_root.duration_p95', 'trace.from_root.error_rate', 'trace.from_root.row_count', 'trace.to_cartservice-0.duration_mean', 'trace.to_cartservice-0.duration_p95', 'trace.to_cartservice-0.error_rate', 'trace.to_cartservice-0.row_count', 'trace.to_cartservice-1.duration_mean', 'trace.to_cartservice-1.duration_p95', 'trace.to_cartservice-1.error_rate', 'trace.to_cartservice-1.row_count', 'trace.to_cartservice-2.duration_mean', 'trace.to_cartservice-2.duration_p95'], 'min_timestamp': 1647750600, 'max_timestamp': 1647752340, 'min_timestamp_shanghai': '2022-03-20T12:30:00+08:00', 'max_timestamp_shanghai': '2022-03-20T12:59:00+08:00', 'top5_rows':     timestamp      cmdb_id                           trace_name      value                ts_shanghai
0  1647750600  adservice-1  trace.from_frontend-1.duration_mean   0.000013  2022-03-20T12:30:00+08:00
1  1647750600  adservice-1   trace.from_frontend-1.duration_p95   0.000014  2022-03-20T12:30:00+08:00
2  1647750600  adservice-1     trace.from_frontend-1.error_rate   0.000000  2022-03-20T12:30:00+08:00
3  1647750600  adservice-1      trace.from_frontend-1.row_count  21.000000  2022-03-20T12:30:00+08:00
4  1647750600  adservice-1  trace.from_frontend-2.duration_mean   0.000013  2022-03-20T12:30:00+08:00}, 'log': {'total_rows': 1814, 'unique_cmdb_count': 31, 'unique_cmdb_list': ['adservice-0', 'adservice-1', 'adservice-2', 'cartservice-0', 'cartservice-1', 'cartservice-2', 'checkoutservice-0', 'checkoutservice-1', 'checkoutservice-2', 'currencyservice-0', 'currencyservice-1', 'currencyservice-2', 'emailservice-0', 'emailservice-1', 'emailservice-2', 'frontend-0', 'frontend-1', 'frontend-2', 'paymentservice-0', 'paymentservice-1', 'paymentservice-2', 'productcatalogservice-0', 'productcatalogservice-1', 'productcatalogservice-2', 'recommendationservice-0', 'recommendationservice-1', 'recommendationservice-2', 'shippingservice-0', 'shippingservice-1', 'shippingservice-2', 'redis-cart-0'], 'unique_name_count': 2, 'unique_name_list': ['log.error_count', 'log.row_count'], 'min_timestamp': 1647750600, 'max_timestamp': 1647752340, 'min_timestamp_shanghai': '2022-03-20T12:30:00+08:00', 'max_timestamp_shanghai': '2022-03-20T12:59:00+08:00', 'top5_rows':     timestamp            cmdb_id         log_name   value                ts_shanghai
0  1647750600        adservice-0    log.row_count  1528.0  2022-03-20T12:30:00+08:00
1  1647750600        adservice-0  log.error_count     0.0  2022-03-20T12:30:00+08:00
2  1647750600  checkoutservice-1    log.row_count    85.0  2022-03-20T12:30:00+08:00
3  1647750600  checkoutservice-1  log.error_count     0.0  2022-03-20T12:30:00+08:00
4  1647750600  checkoutservice-0    log.row_count    79.0  2022-03-20T12:30:00+08:00}, 'error_logs': {'total_rows': 1994, 'unique_cmdb_count': 3, 'unique_cmdb_list': ['frontend-1', 'frontend-2', 'frontend-0'], 'unique_message_count': 1, 'unique_message_list': ['severity: warning, message: failed to retrieve ads'], 'min_timestamp': 1647750600, 'max_timestamp': 1647752399, 'min_timestamp_shanghai': '2022-03-20T12:30:00+08:00', 'max_timestamp_shanghai': '2022-03-20T12:59:59+08:00', 'top5_rows':     timestamp     cmdb_id                                            message                ts_shanghai
0  1647750600  frontend-1  severity: warning, message: failed to retrieve...  2022-03-20T12:30:00+08:00
1  1647750600  frontend-2  severity: warning, message: failed to retrieve...  2022-03-20T12:30:00+08:00
2  1647750603  frontend-0  severity: warning, message: failed to retrieve...  2022-03-20T12:30:03+08:00
3  1647750604  frontend-1  severity: warning, message: failed to retrieve...  2022-03-20T12:30:04+08:00
4  1647750605  frontend-0  severity: warning, message: failed to retrieve...  2022-03-20T12:30:05+08:00}}```
```

In [ ]:
# In[2]:

import pandas as pd
import numpy as np

# reuse df_metric if already loaded in the kernel; otherwise load
try:
    df_metric
except NameError:
    df_metric = pd.read_csv('metric.csv')

# 1) Compute global P95 and P05 per (cmdb_id, kpi_name) using the entire series BEFORE filtering
group = df_metric.groupby(['cmdb_id', 'kpi_name'])['value']
global_p95 = group.quantile(0.95).rename('global_P95')
global_p05 = group.quantile(0.05).rename('global_P05')
global_quantiles = pd.concat([global_p95, global_p05], axis=1).reset_index()

# total number of series inspected
series_count = global_quantiles.shape[0]

# 2) Filter metric.csv rows to the incident time window inclusive
start_ts = 1647750600
end_ts = 1647752340
metric_window = df_metric[(df_metric['timestamp'] >= start_ts) & (df_metric['timestamp'] <= end_ts)].copy()

# 3) Merge thresholds into the windowed data and flag anomalies
mw = metric_window.merge(global_quantiles, on=['cmdb_id', 'kpi_name'], how='left')
mw['high_flag'] = mw['value'] >= mw['global_P95']
mw['low_flag'] = mw['value'] <= mw['global_P05']

# Aggregate per (cmdb_id, kpi_name) in the window
def agg_anomalies(g):
    total_rows = int(g.shape[0])
    high_count = int(g['high_flag'].sum())
    low_count = int(g['low_flag'].sum())
    earliest_high = int(g.loc[g['high_flag'], 'timestamp'].min()) if high_count > 0 else pd.NA
    earliest_low = int(g.loc[g['low_flag'], 'timestamp'].min()) if low_count > 0 else pd.NA
    return pd.Series({
        'total_rows_in_window': total_rows,
        'high_anomaly_count': high_count,
        'low_anomaly_count': low_count,
        'earliest_high_anomaly_ts': earliest_high,
        'earliest_low_anomaly_ts': earliest_low
    })

anomalies_agg = mw.groupby(['cmdb_id', 'kpi_name']).apply(agg_anomalies).reset_index()

# 4) Combine with global thresholds and prepare sorting
result_df = anomalies_agg.merge(global_quantiles, on=['cmdb_id', 'kpi_name'], how='left')

# compute total anomalies and earliest any-anomaly timestamp for sorting
result_df['total_anomalies'] = result_df['high_anomaly_count'] + result_df['low_anomaly_count']
# earliest_any: take min of the two earliest timestamps ignoring NA
result_df['earliest_any_anomaly_ts'] = result_df[['earliest_high_anomaly_ts', 'earliest_low_anomaly_ts']].min(axis=1, skipna=True)

# For sorting, treat missing earliest_any as very large so they appear later
result_df['_earliest_sort'] = result_df['earliest_any_anomaly_ts'].fillna(9999999999)

# sort by total anomalies desc, then earliest anomaly asc
result_df = result_df.sort_values(['total_anomalies', '_earliest_sort'], ascending=[False, True])

# select and format final columns; keep integer nullable dtype for earliest timestamps
final_cols = ['cmdb_id', 'kpi_name', 'total_rows_in_window',
              'high_anomaly_count', 'low_anomaly_count',
              'earliest_high_anomaly_ts', 'earliest_low_anomaly_ts',
              'global_P95', 'global_P05']
final_df = result_df[final_cols].copy()
final_df['earliest_high_anomaly_ts'] = final_df['earliest_high_anomaly_ts'].astype('Int64')
final_df['earliest_low_anomaly_ts'] = final_df['earliest_low_anomaly_ts'].astype('Int64')

# Store intermediate variables in kernel for later steps
global_quantiles, metric_window, mw, anomalies_agg, result_df  # cached

# 5) Return top 20 rows (compact)
top20 = final_df.head(20)

top20, series_count

```
Out[2]:
```
```python
# Summarize the analysis results in plain English.
# Reuse kernel variables from previous steps: top20 (DataFrame) and series_count (int).

try:
    sample_rows = top20.head(5)
    total_series = series_count
except NameError:
    # Fallback if variables not present
    import pandas as pd
    top20 = pd.read_csv('metric.csv').head(0)  # empty placeholder
    sample_rows = top20
    total_series = 'unknown'

# Build concise summary
examples = []
for i, row in sample_rows.iterrows():
    examples.append(f"{row['kpi_name']} (P95={row['global_P95']}, P05={row['global_P05']})")

summary = (
    f"Metric anomaly summary for window 1647750600–1647752340 "
    f"(2022-03-20T12:30:00+08 to 2022-03-20T12:59:00+08):\n"
    f"- Total (cmdb_id, kpi_name) series inspected: {total_series}\n"
    f"- Top 20 series by anomaly count: all top 20 are for cmdb_id 'adservice'.\n"
    f"- For each top series: total_rows_in_window = 30, high_anomaly_count = 30, "
    f"low_anomaly_count = 30, earliest_high_anomaly_ts = 1647750600, "
    f"earliest_low_anomaly_ts = 1647750600.\n"
    f"- Observation: for these top series global P95 == global P05 (see examples), "
    f"so every value in the window equals that threshold and is counted as both a high and low anomaly:\n  "
    + "; ".join(examples)
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(      cmdb_id                                           kpi_name  total_rows_in_window  high_anomaly_count  low_anomaly_count  earliest_high_anomaly_ts  earliest_low_anomaly_ts  global_P95  global_P05
2   adservice                                        app.grpc.rr                    30                  30                 30                1647750600               1647750600       100.0       100.0
4   adservice                                     app.http.count                    30                  30                 30                1647750600               1647750600        24.0        24.0
6   adservice                                        app.http.rr                    30                  30                 30                1647750600               1647750600       100.0       100.0
7   adservice                                        app.http.sr                    30                  30                 30                1647750600               1647750600       100.0       100.0
8   adservice    runtime.java_lang_ClassLoading_LoadedClassCount                    30                  30                 30                1647750600               1647750600      4990.0      4990.0
9   adservice  runtime.java_lang_ClassLoading_TotalLoadedClas...                    30                  30                 30                1647750600               1647750600      4990.0      4990.0
10  adservice  runtime.java_lang_ClassLoading_UnloadedClassCount                    30                  30                 30                1647750600               1647750600         0.0         0.0
11  adservice             runtime.java_lang_ClassLoading_Verbose                    30                  30                 30                1647750600               1647750600         0.0         0.0
12  adservice  runtime.java_lang_Compilation_CompilationTimeM...                    30                  30                 30                1647750600               1647750600         1.0         1.0
15  adservice  runtime.java_lang_GarbageCollector_CollectionC...                    30                  30                 30                1647750600               1647750600         3.0         3.0
17  adservice  runtime.java_lang_GarbageCollector_CollectionT...                    30                  30                 30                1647750600               1647750600       177.0       177.0
18  adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...                    30                  30                 30                1647750600               1647750600         1.0         1.0
19  adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...                    30                  30                 30                1647750600               1647750600         1.0         1.0
21  adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...                    30                  30                 30                1647750600               1647750600        63.0        63.0
23  adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...                    30                  30                 30                1647750600               1647750600     56457.0     56457.0
25  adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...                    30                  30                 30                1647750600               1647750600         3.0         3.0
26  adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...                    30                  30                 30                1647750600               1647750600  22937600.0  22937600.0
27  adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...                    30                  30                 30                1647750600               1647750600   7340032.0   7340032.0
28  adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...                    30                  30                 30                1647750600               1647750600   3801088.0   3801088.0
29  adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...                    30                  30                 30                1647750600               1647750600   3670016.0   3670016.0, 4818)```
```

In [ ]:
# In[3]:

import pandas as pd
import numpy as np
import pytz

tz = pytz.timezone('Asia/Shanghai')

# reuse df_trace if available
try:
    df_trace
except NameError:
    df_trace = pd.read_csv('trace.csv')

# 1) Compute global P95 and P05 per (cmdb_id, trace_name) using the entire series BEFORE filtering
trace_group = df_trace.groupby(['cmdb_id', 'trace_name'])['value']
trace_global_p95 = trace_group.quantile(0.95).rename('global_P95')
trace_global_p05 = trace_group.quantile(0.05).rename('global_P05')
trace_global_quantiles = pd.concat([trace_global_p95, trace_global_p05], axis=1).reset_index()

# total number of (cmdb_id, trace_name) series inspected
trace_series_count = trace_global_quantiles.shape[0]

# 2) Filter trace.csv to the incident time window inclusive
start_ts = 1647750600
end_ts = 1647752340
trace_window = df_trace[(df_trace['timestamp'] >= start_ts) & (df_trace['timestamp'] <= end_ts)].copy()

# 3) Merge thresholds into windowed data and flag anomalies
tw = trace_window.merge(trace_global_quantiles, on=['cmdb_id', 'trace_name'], how='left')
tw['high_flag'] = tw['value'] >= tw['global_P95']
tw['low_flag'] = tw['value'] <= tw['global_P05']

def agg_trace_anomalies(g):
    total_rows = int(g.shape[0])
    high_count = int(g['high_flag'].sum())
    low_count = int(g['low_flag'].sum())
    earliest_high = int(g.loc[g['high_flag'], 'timestamp'].min()) if high_count > 0 else pd.NA
    earliest_low = int(g.loc[g['low_flag'], 'timestamp'].min()) if low_count > 0 else pd.NA
    return pd.Series({
        'total_rows_in_window': total_rows,
        'high_anomaly_count': high_count,
        'low_anomaly_count': low_count,
        'earliest_high_anomaly_ts': earliest_high,
        'earliest_low_anomaly_ts': earliest_low
    })

trace_anomalies_agg = tw.groupby(['cmdb_id', 'trace_name']).apply(agg_trace_anomalies).reset_index()

# 4) Combine with global thresholds and sort
trace_result_df = trace_anomalies_agg.merge(trace_global_quantiles, on=['cmdb_id', 'trace_name'], how='left')
trace_result_df['total_anomalies'] = trace_result_df['high_anomaly_count'] + trace_result_df['low_anomaly_count']
trace_result_df['earliest_any_anomaly_ts'] = trace_result_df[['earliest_high_anomaly_ts','earliest_low_anomaly_ts']].min(axis=1, skipna=True)
trace_result_df['_earliest_sort'] = trace_result_df['earliest_any_anomaly_ts'].fillna(9999999999)
trace_result_df = trace_result_df.sort_values(['total_anomalies', '_earliest_sort'], ascending=[False, True])

# 5) Select final columns and format earliest timestamps as nullable integers
final_cols_trace = ['cmdb_id', 'trace_name', 'total_rows_in_window',
                    'high_anomaly_count', 'low_anomaly_count',
                    'earliest_high_anomaly_ts', 'earliest_low_anomaly_ts',
                    'global_P95', 'global_P05']
final_trace_df = trace_result_df[final_cols_trace].copy()
final_trace_df['earliest_high_anomaly_ts'] = final_trace_df['earliest_high_anomaly_ts'].astype('Int64')
final_trace_df['earliest_low_anomaly_ts'] = final_trace_df['earliest_low_anomaly_ts'].astype('Int64')

# store intermediates in kernel: trace_global_quantiles, trace_window, tw, trace_anomalies_agg, trace_result_df
trace_global_quantiles, trace_window, tw, trace_anomalies_agg, trace_result_df  # cached

# 6) Return compact top 20 rows and series count
top20_trace = final_trace_df.head(20)

top20_trace, trace_series_count

```
Out[3]:
```
```python
# Summarize the trace anomaly results in plain English using existing kernel variables.
from datetime import datetime
import pytz

tz = pytz.timezone('Asia/Shanghai')

try:
    top20_trace  # DataFrame of top 20 from previous step
    trace_series_count
except NameError:
    # Fallback: short note if variables are not present
    summary = "Trace analysis results not found in the kernel."
else:
    # Extract concise observations from the top20 dataframe
    unique_cmdb = top20_trace['cmdb_id'].unique().tolist()
    sample_names = top20_trace['trace_name'].head(6).tolist()
    # common pattern in top rows
    same_rows = (top20_trace['total_rows_in_window'].eq(30).all() and
                 top20_trace['high_anomaly_count'].eq(30).all() and
                 top20_trace['low_anomaly_count'].eq(30).all())
    common_p95_p05_zero = (top20_trace['global_P95'].eq(top20_trace['global_P05']).all() and
                           (top20_trace['global_P95'] == 0.0).any())
    # human-readable earliest timestamp (if present)
    if not top20_trace['earliest_high_anomaly_ts'].isna().all():
        ts_sample = int(top20_trace['earliest_high_anomaly_ts'].dropna().iloc[0])
        ts_hr = datetime.fromtimestamp(ts_sample, tz).isoformat()
    else:
        ts_hr = None

    summary = (
        f"Trace anomaly summary for window 1647750600–1647752340:\n"
        f"- Total (cmdb_id, trace_name) series inspected: {int(trace_series_count)}\n"
        f"- Top 20 results are dominated by these cmdb_id values (examples): {unique_cmdb[:6]}\n"
        f"- Example trace names in top results: {sample_names}\n"
        f"- For the top rows: total_rows_in_window = 30, high_anomaly_count = 30, "
        f"low_anomaly_count = 30, earliest anomaly timestamp ≈ {ts_hr}\n"
        f"- Observation: many top trace series have global_P95 == global_P05 (often 0.0), "
        f"so every point in the window equals that threshold and is counted as both a high and low anomaly.\n"
        f"- Conclusion: the top-ranked trace series reflect a threshold/equality artifact (P95==P05), "
        f"not distinct high vs low deviations across the window."
    )

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(          cmdb_id                                  trace_name  total_rows_in_window  high_anomaly_count  low_anomaly_count  earliest_high_anomaly_ts  earliest_low_anomaly_ts  global_P95  global_P05
2     adservice-0            trace.from_frontend-0.error_rate                    30                  30                 30                1647750600               1647750600         0.0         0.0
6     adservice-0            trace.from_frontend-1.error_rate                    30                  30                 30                1647750600               1647750600         0.0         0.0
10    adservice-0            trace.from_frontend-2.error_rate                    30                  30                 30                1647750600               1647750600         0.0         0.0
14    adservice-1            trace.from_frontend-0.error_rate                    30                  30                 30                1647750600               1647750600         0.0         0.0
18    adservice-1            trace.from_frontend-1.error_rate                    30                  30                 30                1647750600               1647750600         0.0         0.0
22    adservice-1            trace.from_frontend-2.error_rate                    30                  30                 30                1647750600               1647750600         0.0         0.0
26    adservice-2            trace.from_frontend-0.error_rate                    30                  30                 30                1647750600               1647750600         0.0         0.0
30    adservice-2            trace.from_frontend-1.error_rate                    30                  30                 30                1647750600               1647750600         0.0         0.0
34    adservice-2            trace.from_frontend-2.error_rate                    30                  30                 30                1647750600               1647750600         0.0         0.0
38   adservice2-0           trace.from_frontend2-0.error_rate                    30                  30                 30                1647750600               1647750600         0.0         0.0
40  cartservice-0  trace.from_checkoutservice-0.duration_mean                    30                  30                 30                1647750600               1647750600         0.0         0.0
41  cartservice-0   trace.from_checkoutservice-0.duration_p95                    30                  30                 30                1647750600               1647750600         0.0         0.0
42  cartservice-0     trace.from_checkoutservice-0.error_rate                    30                  30                 30                1647750600               1647750600         0.0         0.0
44  cartservice-0  trace.from_checkoutservice-1.duration_mean                    30                  30                 30                1647750600               1647750600         0.0         0.0
45  cartservice-0   trace.from_checkoutservice-1.duration_p95                    30                  30                 30                1647750600               1647750600         0.0         0.0
46  cartservice-0     trace.from_checkoutservice-1.error_rate                    30                  30                 30                1647750600               1647750600         0.0         0.0
48  cartservice-0  trace.from_checkoutservice-2.duration_mean                    30                  30                 30                1647750600               1647750600         0.0         0.0
49  cartservice-0   trace.from_checkoutservice-2.duration_p95                    30                  30                 30                1647750600               1647750600         0.0         0.0
50  cartservice-0     trace.from_checkoutservice-2.error_rate                    30                  30                 30                1647750600               1647750600         0.0         0.0
54  cartservice-0            trace.from_frontend-0.error_rate                    30                  30                 30                1647750600               1647750600         0.0         0.0, 1232)```
```

In [ ]:
# In[4]:

import pandas as pd
import numpy as np
import pytz

# Reuse existing loaded DataFrames if present, otherwise load from CSVs
try:
    df_err
except NameError:
    df_err = pd.read_csv('error_logs.csv')

try:
    df_log
except NameError:
    df_log = pd.read_csv('log.csv')

# Ensure numeric types for value where applicable
if 'value' in df_log.columns:
    df_log['value'] = pd.to_numeric(df_log['value'], errors='coerce')

# Time window (inclusive)
start_ts = 1647750600
end_ts = 1647752340

# 1) error_logs.csv aggregation per cmdb_id
err_agg = (
    df_err.groupby('cmdb_id', dropna=True)
    .agg(
        total_error_log_count=('timestamp', 'size'),
        earliest_error_log_ts=('timestamp', 'min'),
        latest_error_log_ts=('timestamp', 'max')
    )
    .reset_index()
)

# Top 10 by total_error_log_count
error_top10 = err_agg.sort_values('total_error_log_count', ascending=False).head(10).copy()
# Format earliest/latest as nullable integers
error_top10['earliest_error_log_ts'] = error_top10['earliest_error_log_ts'].astype('Int64')
error_top10['latest_error_log_ts'] = error_top10['latest_error_log_ts'].astype('Int64')

# 2) log.csv filtered to window
log_window = df_log[(df_log['timestamp'] >= start_ts) & (df_log['timestamp'] <= end_ts)].copy()

# 2a) log.error_count analysis
log_error_df = log_window[log_window['log_name'] == 'log.error_count'].copy()

# Aggregate per cmdb_id
def agg_log_error(g):
    total_error_count_sum = float(g['value'].sum(skipna=True))
    mask_positive = g['value'] > 0
    minutes_with_error_count = int(mask_positive.sum())
    earliest_minute_with_error_ts = int(g.loc[mask_positive, 'timestamp'].min()) if minutes_with_error_count > 0 else pd.NA
    max_error_value = float(g['value'].max(skipna=True)) if g.shape[0] > 0 else np.nan
    return pd.Series({
        'total_error_count_sum': total_error_count_sum,
        'minutes_with_error_count': minutes_with_error_count,
        'earliest_minute_with_error_ts': earliest_minute_with_error_ts,
        'max_error_value': max_error_value
    })

log_error_agg = log_error_df.groupby('cmdb_id', dropna=True).apply(agg_log_error).reset_index()
log_error_agg['earliest_minute_with_error_ts'] = log_error_agg['earliest_minute_with_error_ts'].astype('Int64')

# Top 20 by minutes_with_error_count desc
log_error_top20 = log_error_agg.sort_values(['minutes_with_error_count','total_error_count_sum'], ascending=[False, False]).head(20).copy()

# 2b) log.row_count analysis
log_row_df = log_window[log_window['log_name'] == 'log.row_count'].copy()

def agg_log_row(g):
    if g.shape[0] == 0:
        return pd.Series({
            'min_row_count_in_window': np.nan,
            'earliest_ts_of_min_row_count': pd.NA,
            'max_row_count_in_window': np.nan
        })
    min_val = float(g['value'].min(skipna=True))
    # earliest timestamp where value == min_val
    mask_min = g['value'] == min_val
    earliest_ts_min = int(g.loc[mask_min, 'timestamp'].min()) if mask_min.any() else pd.NA
    max_val = float(g['value'].max(skipna=True))
    return pd.Series({
        'min_row_count_in_window': min_val,
        'earliest_ts_of_min_row_count': earliest_ts_min,
        'max_row_count_in_window': max_val
    })

log_row_agg = log_row_df.groupby('cmdb_id', dropna=True).apply(agg_log_row).reset_index()
log_row_agg['earliest_ts_of_min_row_count'] = log_row_agg['earliest_ts_of_min_row_count'].astype('Int64')

# Top 20 by min_row_count_in_window ascending (smallest first)
log_row_top20 = log_row_agg.sort_values(['min_row_count_in_window', 'earliest_ts_of_min_row_count'], ascending=[True, True]).head(20).copy()

# 3) Distinct cmdb_id counts
error_logs_distinct_cmdb_count = int(df_err['cmdb_id'].dropna().nunique())
log_window_distinct_cmdb_count = int(log_window['cmdb_id'].dropna().nunique())

# Store intermediates in kernel for later inspection
# err_agg, error_top10, log_window, log_error_df, log_error_agg, log_error_top20, log_row_df, log_row_agg, log_row_top20

err_agg, error_top10, log_window.shape, log_error_agg, log_error_top20, log_row_agg, log_row_top20, error_logs_distinct_cmdb_count, log_window_distinct_cmdb_count

```
Out[4]:
```
Summary of results (compact):

1) error_logs.csv (per cmdb_id)
- Top entries by total_error_log_count (only 3 cmdb_id present):
  - frontend-2: total_error_log_count = 763, earliest = 1647750600, latest = 1647752399
  - frontend-0: total_error_log_count = 695, earliest = 1647750603, latest = 1647752398
  - frontend-1: total_error_log_count = 536, earliest = 1647750600, latest = 1647752398

2) log.csv restricted to timestamps 1647750600–1647752340
a) log.error_count (per cmdb_id)
- Top by minutes_with_error_count:
  - frontend-2: total_error_count_sum = 763.0, minutes_with_error_count = 30, earliest_minute_with_error_ts = 1647750600, max_error_value = 31.0
  - frontend-0: total_error_count_sum = 695.0, minutes_with_error_count = 30, earliest_minute_with_error_ts = 1647750600, max_error_value = 26.0
  - frontend-1: total_error_count_sum = 536.0, minutes_with_error_count = 30, earliest_minute_with_error_ts = 1647750600, max_error_value = 21.0
- Most other cmdb_id have zero error_count (total and minutes_with_error_count = 0).

b) log.row_count (per cmdb_id)
- Smallest min_row_count_in_window (top results sorted ascending):
  - redis-cart-0: min_row_count_in_window = 2.0, earliest_ts_of_min_row_count = 1647751860, max_row_count_in_window = 5.0
  - paymentservice-2: min = 6.0, earliest_ts = 1647752280, max = 19.0
  - paymentservice-1: min = 7.0, earliest_ts = 1647751740, max = 20.0
  - emailservice-0: min = 8.0, earliest_ts = 1647750840, max = 17.0
  - emailservice-1: min = 8.0, earliest_ts = 1647750900, max = 18.0
  - emailservice-2: min = 8.0, earliest_ts = 1647752280, max = 18.0
  - (additional entries follow; full aggregated tables are stored in kernel variables if you want to inspect more)

3) Distinct cmdb_id counts
- error_logs.csv: 3 distinct cmdb_id
- log.csv (within window 1647750600–1647752340): 31 distinct cmdb_id

Notes:
- Intermediate aggregated tables were stored in kernel variables (err_agg, error_top10, log_window, log_error_agg, log_error_top20, log_row_agg, log_row_top20) for later inspection if you want the full top-20 outputs or more rows.

The original code execution output of IPython Kernel is also provided below for reference:

(      cmdb_id  total_error_log_count  earliest_error_log_ts  latest_error_log_ts
0  frontend-0                    695             1647750603           1647752398
1  frontend-1                    536             1647750600           1647752398
2  frontend-2                    763             1647750600           1647752399,       cmdb_id  total_error_log_count  earliest_error_log_ts  latest_error_log_ts
2  frontend-2                    763             1647750600           1647752399
0  frontend-0                    695             1647750603           1647752398
1  frontend-1                    536             1647750600           1647752398, (1814, 4),                     cmdb_id  total_error_count_sum  minutes_with_error_count  earliest_minute_with_error_ts  max_error_value
0               adservice-0                    0.0                       0.0                           <NA>              0.0
1               adservice-1                    0.0                       0.0                           <NA>              0.0
2               adservice-2                    0.0                       0.0                           <NA>              0.0
3             cartservice-0                    0.0                       0.0                           <NA>              0.0
4             cartservice-1                    0.0                       0.0                           <NA>              0.0
..                      ...                    ...                       ...                            ...              ...
26  recommendationservice-2                    0.0                       0.0                           <NA>              0.0
27             redis-cart-0                    0.0                       0.0                           <NA>              0.0
28        shippingservice-0                    0.0                       0.0                           <NA>              0.0
29        shippingservice-1                    0.0                       0.0                           <NA>              0.0
30        shippingservice-2                    0.0                       0.0                           <NA>              0.0

[31 rows x 5 columns],               cmdb_id  total_error_count_sum  minutes_with_error_count  earliest_minute_with_error_ts  max_error_value
17         frontend-2                  763.0                      30.0                     1647750600             31.0
15         frontend-0                  695.0                      30.0                     1647750600             26.0
16         frontend-1                  536.0                      30.0                     1647750600             21.0
0         adservice-0                    0.0                       0.0                           <NA>              0.0
1         adservice-1                    0.0                       0.0                           <NA>              0.0
2         adservice-2                    0.0                       0.0                           <NA>              0.0
3       cartservice-0                    0.0                       0.0                           <NA>              0.0
4       cartservice-1                    0.0                       0.0                           <NA>              0.0
5       cartservice-2                    0.0                       0.0                           <NA>              0.0
6   checkoutservice-0                    0.0                       0.0                           <NA>              0.0
7   checkoutservice-1                    0.0                       0.0                           <NA>              0.0
8   checkoutservice-2                    0.0                       0.0                           <NA>              0.0
9   currencyservice-0                    0.0                       0.0                           <NA>              0.0
10  currencyservice-1                    0.0                       0.0                           <NA>              0.0
11  currencyservice-2                    0.0                       0.0                           <NA>              0.0
12     emailservice-0                    0.0                       0.0                           <NA>              0.0
13     emailservice-1                    0.0                       0.0                           <NA>              0.0
14     emailservice-2                    0.0                       0.0                           <NA>              0.0
18   paymentservice-0                    0.0                       0.0                           <NA>              0.0
19   paymentservice-1                    0.0                       0.0                           <NA>              0.0,                     cmdb_id  min_row_count_in_window  earliest_ts_of_min_row_count  max_row_count_in_window
0               adservice-0                   1354.0                    1647751800                   1618.0
1               adservice-1                    116.0                    1647751920                    148.0
2               adservice-2                    118.0                    1647751800                    148.0
3             cartservice-0                    985.0                    1647751620                   1159.0
4             cartservice-1                    982.0                    1647752340                   1153.0
..                      ...                      ...                           ...                      ...
26  recommendationservice-2                    227.0                    1647752340                    274.0
27             redis-cart-0                      2.0                    1647751860                      5.0
28        shippingservice-0                     81.0                    1647751620                    141.0
29        shippingservice-1                     84.0                    1647751620                    152.0
30        shippingservice-2                     85.0                    1647751620                    142.0

[31 rows x 4 columns],                     cmdb_id  min_row_count_in_window  earliest_ts_of_min_row_count  max_row_count_in_window
27             redis-cart-0                      2.0                    1647751860                      5.0
20         paymentservice-2                      6.0                    1647752280                     19.0
19         paymentservice-1                      7.0                    1647751740                     20.0
12           emailservice-0                      8.0                    1647750840                     17.0
13           emailservice-1                      8.0                    1647750900                     18.0
14           emailservice-2                      8.0                    1647752280                     18.0
18         paymentservice-0                      9.0                    1647751080                     17.0
7         checkoutservice-1                     45.0                    1647752280                    118.0
8         checkoutservice-2                     50.0                    1647752280                     93.0
6         checkoutservice-0                     51.0                    1647751740                    101.0
28        shippingservice-0                     81.0                    1647751620                    141.0
29        shippingservice-1                     84.0                    1647751620                    152.0
30        shippingservice-2                     85.0                    1647751620                    142.0
1               adservice-1                    116.0                    1647751920                    148.0
2               adservice-2                    118.0                    1647751800                    148.0
26  recommendationservice-2                    227.0                    1647752340                    274.0
24  recommendationservice-0                    229.0                    1647752340                    270.0
25  recommendationservice-1                    231.0                    1647752340                    272.0
9         currencyservice-0                    819.0                    1647752280                   1037.0
10        currencyservice-1                    824.0                    1647752280                   1034.0, 3, 31)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[5]:

import pandas as pd
import numpy as np
import pytz

# reuse df_trace and trace_window if available, otherwise load and filter
try:
    df_trace
except NameError:
    df_trace = pd.read_csv('trace.csv')

start_ts = 1647750600
end_ts = 1647752340

try:
    trace_window
except NameError:
    trace_window = df_trace[(df_trace['timestamp'] >= start_ts) & (df_trace['timestamp'] <= end_ts)].copy()

# Ensure numeric value
trace_window['value'] = pd.to_numeric(trace_window['value'], errors='coerce')

# Group 1: trace_name endswith '.row_count'
mask_row_count = trace_window['trace_name'].str.endswith('.row_count', na=False)
trace_row_count = trace_window[mask_row_count].copy()

def agg_row_count(g):
    total_rows = int(g.shape[0])
    if total_rows == 0:
        return pd.Series({
            'min_row_count_in_window': np.nan,
            'earliest_ts_of_min': pd.NA,
            'total_rows_in_window': 0
        })
    min_val = float(g['value'].min(skipna=True))
    mask_min = g['value'] == min_val
    earliest_ts = int(g.loc[mask_min, 'timestamp'].min()) if mask_min.any() else pd.NA
    return pd.Series({
        'min_row_count_in_window': min_val,
        'earliest_ts_of_min': earliest_ts,
        'total_rows_in_window': total_rows
    })

row_count_agg = trace_row_count.groupby(['cmdb_id', 'trace_name'], dropna=True).apply(agg_row_count).reset_index()
row_count_agg['earliest_ts_of_min'] = row_count_agg['earliest_ts_of_min'].astype('Int64')

# Sort ascending by min_row_count_in_window and take top 30
row_count_top30 = row_count_agg.sort_values(['min_row_count_in_window', 'earliest_ts_of_min'], ascending=[True, True]).head(30).copy()

# Count distinct series inspected for row_count group
row_count_series_count = int(row_count_agg.shape[0])

# Group 2: trace_name contains 'error_rate'
mask_error_rate = trace_window['trace_name'].str.contains('error_rate', na=False)
trace_error_rate = trace_window[mask_error_rate].copy()

def agg_error_rate(g):
    total_rows = int(g.shape[0])
    if total_rows == 0:
        return pd.Series({
            'max_error_rate_in_window': np.nan,
            'earliest_ts_of_positive_error': pd.NA,
            'total_rows_in_window': 0
        })
    max_val = float(g['value'].max(skipna=True))
    mask_pos = g['value'] > 0
    earliest_pos = int(g.loc[mask_pos, 'timestamp'].min()) if mask_pos.any() else pd.NA
    return pd.Series({
        'max_error_rate_in_window': max_val,
        'earliest_ts_of_positive_error': earliest_pos,
        'total_rows_in_window': total_rows
    })

error_rate_agg = trace_error_rate.groupby(['cmdb_id', 'trace_name'], dropna=True).apply(agg_error_rate).reset_index()
error_rate_agg['earliest_ts_of_positive_error'] = error_rate_agg['earliest_ts_of_positive_error'].astype('Int64')

# Filter rows where max_error_rate_in_window > 0, sort by earliest positive timestamp asc, limit top 50
error_rate_positive = error_rate_agg[error_rate_agg['max_error_rate_in_window'] > 0].copy()
# Replace NA earliest timestamps with large value for sorting, but keep original column
error_rate_positive['_earliest_sort'] = error_rate_positive['earliest_ts_of_positive_error'].fillna(9999999999)
error_rate_top50 = error_rate_positive.sort_values(['earliest_ts_of_positive_error','_earliest_sort'], ascending=[True, True]).head(50).drop(columns=['_earliest_sort']).copy()

# Count distinct series inspected for error_rate group
error_rate_series_count = int(error_rate_agg.shape[0])

# Store intermediates in kernel variables for follow-up
# row_count_agg, row_count_top30, row_count_series_count, error_rate_agg, error_rate_top50, error_rate_series_count

row_count_top30, error_rate_top50, row_count_series_count, error_rate_series_count

```
Out[5]:
```
Summary of the trace.csv aggregations (window 1647750600–1647752340):

1) trace_name ending with ".row_count"
- Series inspected: 308 distinct (cmdb_id, trace_name) pairs.
- Top 30 (smallest min_row_count_in_window): many series have min_row_count_in_window = 1.0.
- Earliest occurrence of these minima is near the start of the window (timestamps like 1647750600 or 1647750660).
- total_rows_in_window for these top series is typically 29 or 30 (i.e., nearly full-minute coverage in the window).
- Kernel variables saved: row_count_agg, row_count_top30, row_count_series_count.

2) trace_name containing "error_rate"
- Series inspected: 308 distinct (cmdb_id, trace_name) pairs.
- No series had max_error_rate_in_window > 0 in this window — the result set is empty.
- Kernel variables saved: error_rate_agg, error_rate_top50, error_rate_series_count.

If you want, I can (a) show the top30 table with human-readable timestamps, (b) display more rows from row_count_agg, or (c) re-check specific services/pods.

The original code execution output of IPython Kernel is also provided below for reference:

(               cmdb_id                              trace_name  min_row_count_in_window  earliest_ts_of_min  total_rows_in_window
34   checkoutservice-0         trace.from_frontend-0.row_count                      1.0          1647750600                  29.0
47   checkoutservice-0       trace.to_emailservice-2.row_count                      1.0          1647750600                  30.0
49   checkoutservice-0     trace.to_paymentservice-1.row_count                      1.0          1647750600                  30.0
58   checkoutservice-1         trace.from_frontend-1.row_count                      1.0          1647750600                  29.0
69   checkoutservice-1       trace.to_emailservice-1.row_count                      1.0          1647750600                  29.0
..                 ...                                     ...                      ...                 ...                   ...
72   checkoutservice-1     trace.to_paymentservice-1.row_count                      1.0          1647750660                  29.0
79   checkoutservice-1    trace.to_shippingservice-2.row_count                      1.0          1647750660                  30.0
133     emailservice-0  trace.from_checkoutservice-1.row_count                      1.0          1647750660                  29.0
135     emailservice-1  trace.from_checkoutservice-0.row_count                      1.0          1647750660                  28.0
139     emailservice-2  trace.from_checkoutservice-1.row_count                      1.0          1647750660                  30.0

[30 rows x 5 columns], Empty DataFrame
Columns: [cmdb_id, trace_name, max_error_rate_in_window, earliest_ts_of_positive_error, total_rows_in_window]
Index: [], 308, 308)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[6]:

import pandas as pd
import numpy as np

# Reuse existing trace_window and trace_row_count if available; otherwise compute
try:
    trace_window
except NameError:
    df_trace = pd.read_csv('trace.csv')
    start_ts = 1647750600
    end_ts = 1647752340
    trace_window = df_trace[(df_trace['timestamp'] >= start_ts) & (df_trace['timestamp'] <= end_ts)].copy()

# ensure numeric
trace_window['value'] = pd.to_numeric(trace_window['value'], errors='coerce')

try:
    trace_row_count
except NameError:
    mask_row_count = trace_window['trace_name'].str.endswith('.row_count', na=False)
    trace_row_count = trace_window[mask_row_count].copy()

# selection mask per requirements
mask_selection = (
    trace_row_count['cmdb_id'].str.startswith('adservice', na=False)
    | trace_row_count['trace_name'].str.contains('from_frontend', na=False)
    | trace_row_count['trace_name'].str.contains('to_adservice', na=False)
    | trace_row_count['cmdb_id'].str.contains('redis-cart', na=False)
    | trace_row_count['cmdb_id'].str.contains('paymentservice', na=False)
    | trace_row_count['cmdb_id'].str.contains('recommendationservice', na=False)
)

trace_row_selected = trace_row_count[mask_selection].copy()

# aggregate per (cmdb_id, trace_name)
def agg_selected(g):
    total_rows = int(g.shape[0])
    min_val = float(g['value'].min(skipna=True))
    mask_min = g['value'] == min_val
    earliest_min = int(g.loc[mask_min, 'timestamp'].min()) if mask_min.any() else pd.NA
    return pd.Series({
        'min_row_count_in_window': min_val,
        'earliest_ts_of_min': earliest_min,
        'total_rows_in_window': total_rows
    })

selected_agg = trace_row_selected.groupby(['cmdb_id', 'trace_name'], dropna=True).apply(agg_selected).reset_index()
selected_agg['earliest_ts_of_min'] = selected_agg['earliest_ts_of_min'].astype('Int64')
selected_agg['total_rows_in_window'] = selected_agg['total_rows_in_window'].astype('Int64')

# sort by earliest_ts_of_min ascending and limit top 50
selected_top50 = selected_agg.sort_values(['earliest_ts_of_min', 'min_row_count_in_window'], ascending=[True, True]).head(50).copy()

# distinct cmdb_id values and series count
distinct_cmdbs = pd.DataFrame({'cmdb_id': sorted(selected_agg['cmdb_id'].dropna().unique())})
series_count = int(selected_agg.shape[0])

# Store intermediates for follow-up: trace_row_selected, selected_agg, selected_top50, distinct_cmdbs, series_count

selected_top50, distinct_cmdbs, series_count

```
Out[6]:
```
Summary of the requested filtered trace.row_count analysis (window 1647750600–1647752340):

- Series inspected:
  - Total (cmdb_id, trace_name) series in this subset: 106.

- Distinct cmdb_id values found in the subset:
  - 36 distinct cmdb_id values (examples): adservice-0, adservice-1, adservice-2, adservice2-0, cartservice-0, checkoutservice-0, checkoutservice-1, checkoutservice-2, paymentservice-0, recommendationservice-1, recommendationservice-2, recommendationservice2-0, shippingservice-0, shippingservice-1, shippingservice-2, … (36 total).

- Top results (sorted by earliest timestamp of the minimum row_count, earliest first). Example rows from the top of the returned table (all values are "min_row_count_in_window", "earliest_ts_of_min", "total_rows_in_window"):
  1. checkoutservice-0 — trace.from_frontend-0.row_count: min = 1.0, earliest_ts_of_min = 1647750600, total_rows = 29
  2. checkoutservice-1 — trace.from_frontend-1.row_count: min = 1.0, earliest_ts_of_min = 1647750600, total_rows = 29
  3. checkoutservice-2 — trace.from_frontend-1.row_count: min = 1.0, earliest_ts_of_min = 1647750600, total_rows = 25
  4. paymentservice-0 — trace.from_checkoutservice-1.row_count: min = 1.0, earliest_ts_of_min = 1647750600, total_rows = 30
  5. paymentservice-0 — trace.from_checkoutservice-2.row_count: min = 1.0, earliest_ts_of_min = 1647750600, total_rows = 30
  ...
  - Later examples in the top-50 include recommendationservice-1 and recommendationservice-2 entries with min_row_count_in_window = 17.0 at earliest_ts 1647751020, and many other services with later earliest minima.

- Observations:
  - Many selected series reach their minimum row_count very early in the window (timestamp 1647750600 or nearby).
  - Several services (checkoutservice, paymentservice, recommendationservice, adservice-related entries, etc.) appear in this filtered subset.
  - The returned result set was limited to the top 50 rows sorted by earliest_ts_of_min ascending.

- Follow-up / kernel variables (saved for inspection):
  - trace_row_selected (filtered rows)
  - selected_agg (aggregated per series)
  - selected_top50 (final top-50 table)
  - distinct_cmdbs (distinct cmdb_id list)
  - series_count (=106)

If you want, I can (a) show the top-50 table with human-readable timestamps, (b) display the full distinct cmdb_id list, or (c) export specific series for investigation. Which would you like next?

The original code execution output of IPython Kernel is also provided below for reference:

(                      cmdb_id                                  trace_name  min_row_count_in_window  earliest_ts_of_min  total_rows_in_window
20          checkoutservice-0             trace.from_frontend-0.row_count                      1.0          1647750600                    29
24          checkoutservice-1             trace.from_frontend-1.row_count                      1.0          1647750600                    29
27          checkoutservice-2             trace.from_frontend-1.row_count                      1.0          1647750600                    25
51           paymentservice-0      trace.from_checkoutservice-1.row_count                      1.0          1647750600                    30
52           paymentservice-0      trace.from_checkoutservice-2.row_count                      1.0          1647750600                    30
..                        ...                                         ...                      ...                 ...                   ...
102         shippingservice-2             trace.from_frontend-0.row_count                      6.0          1647751020                    30
78    recommendationservice-1             trace.from_frontend-1.row_count                     17.0          1647751020                    30
85    recommendationservice-2             trace.from_frontend-1.row_count                     17.0          1647751020                    30
11              cartservice-0             trace.from_frontend-1.row_count                     23.0          1647751020                    30
93   recommendationservice2-0  trace.to_productcatalogservice-0.row_count                     13.0          1647751140                    30

[50 rows x 5 columns],                      cmdb_id
0                adservice-0
1                adservice-1
2                adservice-2
3               adservice2-0
4              cartservice-0
..                       ...
31  recommendationservice2-0
32         shippingservice-0
33         shippingservice-1
34         shippingservice-2
35        shippingservice2-0

[36 rows x 1 columns], 106)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[7]:

import pandas as pd
import numpy as np
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse trace_window if available, otherwise load and filter df_trace
try:
    trace_window
except NameError:
    df_trace = pd.read_csv('trace.csv')
    start_ts = 1647750600
    end_ts = 1647752340
    trace_window = df_trace[(df_trace['timestamp'] >= start_ts) & (df_trace['timestamp'] <= end_ts)].copy()

# Ensure numeric values
trace_window['value'] = pd.to_numeric(trace_window['value'], errors='coerce')

# Selection: cmdb_id startswith 'adservice' OR trace_name contains 'adservice'
mask_sel = (
    trace_window['cmdb_id'].str.startswith('adservice', na=False)
    | trace_window['trace_name'].str.contains('adservice', na=False)
)
trace_selected = trace_window[mask_sel].copy()

# Aggregate per (cmdb_id, trace_name)
def agg_adservice(g):
    total_rows = int(g.shape[0])
    min_val = float(g['value'].min(skipna=True)) if total_rows > 0 else np.nan
    mask_min = g['value'] == min_val
    earliest_min = int(g.loc[mask_min, 'timestamp'].min()) if mask_min.any() else pd.NA
    return pd.Series({
        'min_row_count_in_window': min_val,
        'earliest_ts_of_min': earliest_min,
        'total_rows_in_window': total_rows
    })

adservice_agg = trace_selected.groupby(['cmdb_id', 'trace_name'], dropna=True).apply(agg_adservice).reset_index()
adservice_agg['earliest_ts_of_min'] = adservice_agg['earliest_ts_of_min'].astype('Int64')
adservice_agg['total_rows_in_window'] = adservice_agg['total_rows_in_window'].astype('Int64')

# Sort by earliest_ts_of_min ascending and take top 20
adservice_top20 = adservice_agg.sort_values(['earliest_ts_of_min', 'min_row_count_in_window'], ascending=[True, True]).head(20).copy()

# Distinct cmdb_id values and series count
distinct_cmdbs_adservice = pd.DataFrame({'cmdb_id': sorted(adservice_agg['cmdb_id'].dropna().unique())})
adservice_series_count = int(adservice_agg.shape[0])

# Store intermediates for follow-up: trace_selected, adservice_agg, adservice_top20, distinct_cmdbs_adservice, adservice_series_count

adservice_top20, distinct_cmdbs_adservice, adservice_series_count

```
Out[7]:
```
Summary of the requested extraction (trace.csv, window 1647750600–1647752340; rows where cmdb_id starts with "adservice" OR trace_name contains "adservice"):

- Series inspected: 80 distinct (cmdb_id, trace_name) pairs.
- Distinct cmdb_id values found (examples / full set):  
  adservice-0, adservice-1, adservice-2, adservice2-0, frontend-0, frontend-1, frontend-2, frontend2-0.
- Top 20 rows (sorted by earliest timestamp of the minimum) show the following pattern:
  - Many traces (e.g., trace.from_frontend-*.error_rate and trace.to_adservice-*.error_rate) have min_row_count_in_window = 0.0.
  - For these top rows the earliest_ts_of_min = 1647750600 (the start of the incident window).
  - total_rows_in_window is 30 for these entries (full-minute coverage in the window).
- Interpretation: for the selected adservice-related traces, the minimum observed value in the window is zero and it occurs at the very start of the window for many series; this was observed across the 80 series returned.

Kernel variables saved for follow-up: trace_selected, adservice_agg, adservice_top20, distinct_cmdbs_adservice, adservice_series_count.

The original code execution output of IPython Kernel is also provided below for reference:

(         cmdb_id                         trace_name  min_row_count_in_window  earliest_ts_of_min  total_rows_in_window
2    adservice-0   trace.from_frontend-0.error_rate                      0.0          1647750600                    30
6    adservice-0   trace.from_frontend-1.error_rate                      0.0          1647750600                    30
10   adservice-0   trace.from_frontend-2.error_rate                      0.0          1647750600                    30
14   adservice-1   trace.from_frontend-0.error_rate                      0.0          1647750600                    30
18   adservice-1   trace.from_frontend-1.error_rate                      0.0          1647750600                    30
22   adservice-1   trace.from_frontend-2.error_rate                      0.0          1647750600                    30
26   adservice-2   trace.from_frontend-0.error_rate                      0.0          1647750600                    30
30   adservice-2   trace.from_frontend-1.error_rate                      0.0          1647750600                    30
34   adservice-2   trace.from_frontend-2.error_rate                      0.0          1647750600                    30
38  adservice2-0  trace.from_frontend2-0.error_rate                      0.0          1647750600                    30
42    frontend-0    trace.to_adservice-0.error_rate                      0.0          1647750600                    30
46    frontend-0    trace.to_adservice-1.error_rate                      0.0          1647750600                    30
50    frontend-0    trace.to_adservice-2.error_rate                      0.0          1647750600                    30
54    frontend-1    trace.to_adservice-0.error_rate                      0.0          1647750600                    30
58    frontend-1    trace.to_adservice-1.error_rate                      0.0          1647750600                    30
62    frontend-1    trace.to_adservice-2.error_rate                      0.0          1647750600                    30
66    frontend-2    trace.to_adservice-0.error_rate                      0.0          1647750600                    30
70    frontend-2    trace.to_adservice-1.error_rate                      0.0          1647750600                    30
74    frontend-2    trace.to_adservice-2.error_rate                      0.0          1647750600                    30
78   frontend2-0   trace.to_adservice2-0.error_rate                      0.0          1647750600                    30,         cmdb_id
0   adservice-0
1   adservice-1
2   adservice-2
3  adservice2-0
4    frontend-0
5    frontend-1
6    frontend-2
7   frontend2-0, 80)```
```

In [ ]:
# In[8]:

import pandas as pd
import numpy as np
import pytz

tz = pytz.timezone('Asia/Shanghai')

# reuse df_metric if present
try:
    df_metric
except NameError:
    df_metric = pd.read_csv('metric.csv')

# Ensure numeric
df_metric['value'] = pd.to_numeric(df_metric['value'], errors='coerce')

# Filter to window and cmdb_id starting with 'adservice'
start_ts = 1647750600
end_ts = 1647752340
mask_time = (df_metric['timestamp'] >= start_ts) & (df_metric['timestamp'] <= end_ts)
mask_cmdb = df_metric['cmdb_id'].str.startswith('adservice', na=False)
mask_kpi = df_metric['kpi_name'].isin(['app.http.count', 'app.http.sr'])

adservice_http_window = df_metric[mask_time & mask_cmdb & mask_kpi].copy()

# Aggregate per (cmdb_id, kpi_name)
def agg_http(g):
    total_rows = int(g.shape[0])
    if total_rows == 0:
        return pd.Series({
            'total_rows_in_window': 0,
            'min_value_in_window': np.nan,
            'max_value_in_window': np.nan,
            'earliest_ts_of_min': pd.NA,
            'earliest_ts_of_max': pd.NA
        })
    min_val = float(g['value'].min(skipna=True))
    max_val = float(g['value'].max(skipna=True))
    mask_min = g['value'] == min_val
    mask_max = g['value'] == max_val
    earliest_min = int(g.loc[mask_min, 'timestamp'].min()) if mask_min.any() else pd.NA
    earliest_max = int(g.loc[mask_max, 'timestamp'].min()) if mask_max.any() else pd.NA
    return pd.Series({
        'total_rows_in_window': total_rows,
        'min_value_in_window': min_val,
        'max_value_in_window': max_val,
        'earliest_ts_of_min': earliest_min,
        'earliest_ts_of_max': earliest_max
    })

adservice_http_agg = adservice_http_window.groupby(['cmdb_id', 'kpi_name'], dropna=True).apply(agg_http).reset_index()
adservice_http_agg['earliest_ts_of_min'] = adservice_http_agg['earliest_ts_of_min'].astype('Int64')
adservice_http_agg['earliest_ts_of_max'] = adservice_http_agg['earliest_ts_of_max'].astype('Int64')
adservice_http_agg['total_rows_in_window'] = adservice_http_agg['total_rows_in_window'].astype('Int64')

# Sort by earliest_ts_of_min ascending and limit top 20
adservice_http_agg_top20 = adservice_http_agg.sort_values(['earliest_ts_of_min', 'min_value_in_window'], ascending=[True, True]).head(20).copy()

# Number of distinct (cmdb_id, kpi_name) series inspected
adservice_http_series_count = int(adservice_http_agg.shape[0])

# Store intermediates for follow-up: adservice_http_window, adservice_http_agg, adservice_http_agg_top20, adservice_http_series_count
adservice_http_agg_top20, adservice_http_series_count

```
Out[8]:
```
```python
# Summarize the previous metric aggregation result for adservice HTTP KPIs.
from datetime import datetime
import pytz

tz = pytz.timezone('Asia/Shanghai')

try:
    ad_df = adservice_http_agg_top20  # from prior kernel state
    series_count = int(adservice_http_series_count)
except NameError:
    # Fallback if variables not present
    import pandas as pd
    ad_df = pd.DataFrame([
        {'cmdb_id':'adservice','kpi_name':'app.http.count','total_rows_in_window':30,'min_value_in_window':24.0,'max_value_in_window':24.0,'earliest_ts_of_min':1647750600,'earliest_ts_of_max':1647750600},
        {'cmdb_id':'adservice','kpi_name':'app.http.sr','total_rows_in_window':30,'min_value_in_window':100.0,'max_value_in_window':100.0,'earliest_ts_of_min':1647750600,'earliest_ts_of_max':1647750600},
    ])
    series_count = 2

def fmt_ts(ts):
    try:
        return datetime.fromtimestamp(int(ts), tz).isoformat()
    except Exception:
        return 'null'

lines = []
lines.append("Metric summary for adservice KPIs within 1647750600–1647752340 (2022-03-20T12:30:00+08 to 2022-03-20T12:59:00+08):")
lines.append(f"- Distinct (cmdb_id,kpi_name) series inspected: {series_count}")
for _, r in ad_df.iterrows():
    cmdb = r['cmdb_id']
    kpi = r['kpi_name']
    total = int(r['total_rows_in_window'])
    mn = r['min_value_in_window']
    mx = r['max_value_in_window']
    et_min = fmt_ts(r['earliest_ts_of_min'])
    et_max = fmt_ts(r['earliest_ts_of_max'])
    lines.append(f"- {cmdb} / {kpi}: total_rows_in_window={total}, min={mn}, max={mx}, earliest_min={et_min}, earliest_max={et_max}")

lines.append("- Observation: for both KPIs all values in the window are identical (min == max) and occur at the start of the window.")

summary = "\n".join(lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(     cmdb_id        kpi_name  total_rows_in_window  min_value_in_window  max_value_in_window  earliest_ts_of_min  earliest_ts_of_max
0  adservice  app.http.count                    30                 24.0                 24.0          1647750600          1647750600
1  adservice     app.http.sr                    30                100.0                100.0          1647750600          1647750600, 2)```
```

In [ ]:
# In[9]:

import pandas as pd
import numpy as np
import pytz

# Reuse existing variables if present; otherwise compute them
try:
    trace_window
except NameError:
    df_trace = pd.read_csv('trace.csv')
    start_ts = 1647750600
    end_ts = 1647752340
    trace_window = df_trace[(df_trace['timestamp'] >= start_ts) & (df_trace['timestamp'] <= end_ts)].copy()

# Ensure numeric values
trace_window['value'] = pd.to_numeric(trace_window['value'], errors='coerce')

# Ensure we have the row_count subset cached
try:
    trace_row_count
except NameError:
    mask_row_count = trace_window['trace_name'].str.endswith('.row_count', na=False)
    trace_row_count = trace_window[mask_row_count].copy()

# 1) Group by cmdb_id and compute requested aggregates across all .row_count traces for that cmdb_id
def agg_per_cmdb(g):
    total_rows = int(g.shape[0])
    if total_rows == 0:
        return pd.Series({
            'min_row_count_in_window': np.nan,
            'earliest_ts_of_min': pd.NA,
            'total_rows_in_window': 0
        })
    min_val = float(g['value'].min(skipna=True))
    mask_min = g['value'] == min_val
    earliest_min = int(g.loc[mask_min, 'timestamp'].min()) if mask_min.any() else pd.NA
    return pd.Series({
        'min_row_count_in_window': min_val,
        'earliest_ts_of_min': earliest_min,
        'total_rows_in_window': total_rows
    })

cmdb_agg = trace_row_count.groupby('cmdb_id', dropna=True).apply(agg_per_cmdb).reset_index()
cmdb_agg['earliest_ts_of_min'] = cmdb_agg['earliest_ts_of_min'].astype('Int64')
cmdb_agg['total_rows_in_window'] = cmdb_agg['total_rows_in_window'].astype('Int64')

# 2) Exclude cmdb_id values that start with 'frontend'
mask_not_frontend = ~cmdb_agg['cmdb_id'].str.startswith('frontend', na=False)
cmdb_agg_no_frontend = cmdb_agg[mask_not_frontend].copy()

# 3) From remaining groups, select rows where earliest_ts_of_min > 1647750700
threshold_ts = 1647750700
cmdb_late_min = cmdb_agg_no_frontend[
    cmdb_agg_no_frontend['earliest_ts_of_min'].notna() &
    (cmdb_agg_no_frontend['earliest_ts_of_min'] > threshold_ts)
].copy()

# 4) Sort by earliest_ts_of_min ascending and limit to top 20
cmdb_rowcount_late_top20 = cmdb_late_min.sort_values('earliest_ts_of_min', ascending=True).head(20).reset_index(drop=True)

# Number of series inspected in this filtered set (count of cmdb_id groups meeting the condition)
late_series_count = int(cmdb_late_min.shape[0])

# Store intermediates for follow-up: cmdb_agg, cmdb_agg_no_frontend, cmdb_late_min, cmdb_rowcount_late_top20, late_series_count

cmdb_rowcount_late_top20, late_series_count

```
Out[9]:
```
```python
from datetime import datetime
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse kernel variables: cmdb_rowcount_late_top20, late_series_count
try:
    df = cmdb_rowcount_late_top20.copy()
    count = int(late_series_count)
except NameError:
    df = None
    count = 0

def to_shanghai(ts):
    try:
        return datetime.fromtimestamp(int(ts), tz).isoformat()
    except Exception:
        return 'null'

if df is None or df.shape[0] == 0:
    summary = "No matching series found in the filtered trace.row_count data."
else:
    lines = []
    lines.append(f"Trace.row_count per-cmdb minima occurring after 1647750700 (post-12:31):")
    lines.append(f"- Number of cmdb_id groups meeting the condition: {count}")
    lines.append("- Top results (sorted by earliest_ts_of_min ascending), examples:")
    # show up to 8 example rows
    n_show = min(8, df.shape[0])
    for i, row in df.head(n_show).iterrows():
        cmdb = row['cmdb_id']
        mn = row['min_row_count_in_window']
        ets = to_shanghai(row['earliest_ts_of_min'])
        total = int(row['total_rows_in_window'])
        lines.append(f"  • {cmdb}: min={mn}, earliest_min={ets}, total_rows={total}")
    lines.append("- Note: totals reflect aggregated counts across all '.row_count' traces for each cmdb_id.")
    summary = "\n".join(lines)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(                     cmdb_id  min_row_count_in_window  earliest_ts_of_min  total_rows_in_window
0    productcatalogservice-0                      1.0          1647750720                   300
1               adservice2-0                     28.0          1647750780                    30
2              cartservice-1                      1.0          1647750840                   210
3         checkoutservice2-0                      1.0          1647750960                   261
4          paymentservice2-0                      1.0          1647750960                    29
5            emailservice2-0                      1.0          1647750960                    29
6             cartservice2-0                      2.0          1647750960                    89
7         shippingservice2-0                      2.0          1647750960                    59
8    recommendationservice-2                     17.0          1647751020                   210
9          shippingservice-1                      1.0          1647751020                   180
10   recommendationservice-1                     17.0          1647751020                   210
11             cartservice-2                      1.0          1647751020                   210
12  recommendationservice2-0                     13.0          1647751140                   150
13  productcatalogservice2-0                      1.0          1647751200                    59
14        currencyservice2-0                      2.0          1647751200                    59
15             cartservice-0                      1.0          1647751380                   210
16         currencyservice-0                      2.0          1647751740                   180
17   productcatalogservice-2                      1.0          1647751860                   300
18         currencyservice-2                      2.0          1647751860                   180
19   productcatalogservice-1                      1.0          1647751860                   300, 22)```
```